# 01 — Analyse exploratoire (EDA)

Données brutes `data/raw/application_train.csv` (Home Credit). Objectif : comprendre la cible, les valeurs manquantes, les anomalies et les groupes utiles pour l'équité (D5) et le thin file (D10).

L'EDA porte sur les **307 511 lignes**, y compris les futures lignes de test : c'est descriptif, aucun choix ajusté (seuil, paramètre, sélection de features) n'en dépend. Les figures ne s'affichent pas dans le notebook (backend `Agg`) : elles sont enregistrées en PNG dans `reports/figures/`.

In [ ]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from crediscore.config import load_params

matplotlib.use("Agg")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
params = load_params(ROOT / "params.yaml")

# Chart style: one hue, light surface, recessive axes.
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": GREY, "axes.labelcolor": INK, "xtick.color": INK,
    "ytick.color": INK, "text.color": INK, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 9,
})


def save(fig: plt.Figure, name: str) -> None:
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=100)
    plt.close(fig)
    print(f"{path.name}: {path.stat().st_size / 1024:.0f} KB")


def rate_bars(rates: pd.Series, counts: pd.Series, overall: float,
              title: str, name: str) -> None:
    """Horizontal bars of default rate per group, overall rate as reference."""
    fig, ax = plt.subplots(figsize=(6, 1.0 + 0.45 * len(rates)))
    ax.barh(rates.index.astype(str), rates * 100, color=BLUE, height=0.55)
    ax.axvline(overall * 100, color=GREY, ls="--", lw=1)
    for i, (r, n) in enumerate(zip(rates, counts)):
        ax.text(r * 100, i, f" {r:.1%} (n={n:,})".replace(",", " "), va="center",
                bbox={"fc": "#fcfcfb", "ec": "none", "pad": 1})
    ax.set_xlabel("Taux de défaut (%)")
    ax.set_title(f"{title}\n(pointillé : moyenne {overall:.1%})", loc="left")
    ax.invert_yaxis()
    ax.set_xlim(0, rates.max() * 100 * 1.45)
    save(fig, name)

In [ ]:
df = pd.read_csv(ROOT / params.data.raw_path)
target = params.target
print(df.shape)

## 1. Équilibre des classes

In [ ]:
counts = df[target].value_counts().sort_index()
default_rate = df[target].mean()
print(counts.to_dict(), f"default rate = {default_rate:.2%}")

fig, ax = plt.subplots(figsize=(5, 1.8))
labels = ["0 — rembourse", "1 — défaut"]
ax.barh(labels, counts.values, color=BLUE, height=0.55)
for i, n in enumerate(counts.values):
    ax.text(n, i, f" {n:,} ({n / len(df):.1%})".replace(",", " "), va="center")
ax.set_xlim(0, counts.max() * 1.35)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
ax.set_title("Répartition de TARGET", loc="left")
ax.invert_yaxis()
save(fig, "eda_class_balance.png")

**Classe minoritaire : 8,07 %** (24 825 défauts sur 307 511). L'accuracy n'a donc aucun sens (91,9 % en prédisant toujours 0) : on suivra ROC-AUC, PR-AUC, Brier et le coût métier, avec `class_weight="balanced"` (T5).

## 2. Valeurs manquantes

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print("columns with missing values:", (missing > 0).sum(), "/", df.shape[1])
print(f"columns above missing_threshold ({params.data.missing_threshold}):",
      (missing > params.data.missing_threshold).sum())
print("model features with missing values:")
print(missing.reindex(params.features.all).dropna()[lambda s: s > 0]
      .sort_values(ascending=False).round(3).to_string())

top = missing.head(20)[::-1]
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(top.index, top * 100, color=BLUE, height=0.6)
ax.axvline(params.data.missing_threshold * 100, color=GREY, ls="--", lw=1)
ax.set_xlabel("Valeurs manquantes (%)")
ax.set_title("20 colonnes les plus incomplètes\n(pointillé : seuil de 60 %)",
             loc="left")
save(fig, "eda_missing_top20.png")

**67 colonnes sur 122** ont des manquants ; **17** dépassent le seuil de 60 % (surtout les descriptifs du logement `*_AVG/_MODE/_MEDI`, jusqu'à 69,9 %). Parmi nos features, les plus incomplètes sont `EXT_SOURCE_1` (56 %), `TOTALAREA_MODE` (48 %) et `OCCUPATION_TYPE` (31 %) : l'imputation est indispensable.

## 3. `DAYS_EMPLOYED = 365243` (valeur sentinelle)

In [ ]:
anom = df["DAYS_EMPLOYED"] == 365243
print(f"anomalous rows: {anom.sum():,} ({anom.mean():.1%})")
print(f"default rate anomalous = {df.loc[anom, target].mean():.2%}, "
      f"others = {df.loc[~anom, target].mean():.2%}")
print(df.loc[anom, "NAME_INCOME_TYPE"].value_counts().to_string())

**55 374 dossiers (18,0 %)** ont `DAYS_EMPLOYED = 365243` (≈ 1 000 ans) : ce sont uniquement des retraités et des sans-emploi. Leur taux de défaut est plus bas (**5,4 %** contre 8,7 %). On ne supprime pas ces lignes : la valeur devient NaN dans `EMPLOYED_YEARS` et l'indicateur `DAYS_EMPLOYED_ANOM` garde l'information.

## 4. Corrélations avec `TARGET`

In [ ]:
num = df.select_dtypes("number").drop(columns=[target, params.id_col])
num["DAYS_EMPLOYED"] = num["DAYS_EMPLOYED"].replace(365243, np.nan)
corr = num.corrwith(df[target]).dropna()
top_corr = corr.reindex(corr.abs().sort_values(ascending=False).index).head(15)
print(top_corr.round(3).to_string())

s = top_corr[::-1]
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(s.index, s, color=[ORANGE if v > 0 else BLUE for v in s], height=0.6)
for i, v in enumerate(s):
    ax.text(v, i, f" {v:+.3f} " , va="center", ha="left" if v > 0 else "right")
ax.axvline(0, color=GREY, lw=1)
ax.set_xlim(-0.26, 0.14)
ax.set_xlabel("Corrélation de Pearson avec TARGET")
ax.set_title("15 plus fortes corrélations avec le défaut\n"
             "(bleu : protège, orange : risque)", loc="left")
save(fig, "eda_top_correlations.png")

Les trois scores externes dominent : **`EXT_SOURCE_3` −0,179, `EXT_SOURCE_2` −0,160, `EXT_SOURCE_1` −0,155**. Viennent ensuite l'âge (`DAYS_BIRTH` +0,078, donc les jeunes font plus défaut), l'ancienneté d'emploi (+0,075) et la note de la région (+0,061). Les corrélations linéaires restent faibles : il faudra combiner beaucoup de signaux.

## 5. Manquants des `EXT_SOURCE_*`

In [ ]:
ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
rows = []
for c in ext:
    m = df[c].isna()
    rows.append({"feature": c, "missing_rate": m.mean(),
                 "default_if_missing": df.loc[m, target].mean(),
                 "default_if_present": df.loc[~m, target].mean()})
ext_table = pd.DataFrame(rows).set_index("feature")
print(ext_table.round(4).to_string())
n_missing = df[ext].isna().sum(axis=1)
print(df.groupby(n_missing)[target].agg(["size", "mean"]).round(4).to_string())

`EXT_SOURCE_1` manque dans **56,4 %** des dossiers, `EXT_SOURCE_3` dans **19,8 %**, `EXT_SOURCE_2` presque jamais (0,2 %). L'absence est elle-même un signal : quand `EXT_SOURCE_3` manque, le taux de défaut passe à 9,3 % (7,8 % sinon), d'où la feature `EXT_SOURCE_MISSING_COUNT`.

## 6. Segment thin file (D10 : `EXT_SOURCE_1` **et** `EXT_SOURCE_3` manquants)

In [ ]:
thin = df["EXT_SOURCE_1"].isna() & df["EXT_SOURCE_3"].isna()
print(f"thin file: {thin.sum():,} rows ({thin.mean():.1%}), "
      f"default rate {df.loc[thin, target].mean():.2%} "
      f"vs {df.loc[~thin, target].mean():.2%} for the others")
print(f"good payers in the thin-file segment: {(thin & (df[target] == 0)).sum():,}")

**36 574 dossiers (11,9 %)** sont thin file. Leur taux de défaut est **9,95 %** contre 7,8 % pour les autres : un peu plus risqués, mais **90 % d'entre eux remboursent** (32 936 bons payeurs). La politique « scorecard classique » (refus de tous les thin files, D10) rejetterait tous ces bons payeurs : c'est la référence du KPI inclusion.

## 7. Taux de défaut par sexe et par tranche d'âge (équité, D5)

In [ ]:
gender = df.groupby("CODE_GENDER")[target].agg(["size", "mean"])
print(gender.round(4).to_string())
g = gender.drop(index="XNA")
rate_bars(g["mean"], g["size"], default_rate,
          "Taux de défaut par CODE_GENDER (XNA exclu, 4 lignes)",
          "eda_default_by_gender.png")

age = -df["DAYS_BIRTH"] / 365.25
band = pd.cut(age, [0, 30, 50, np.inf], right=False, labels=["<30", "30-50", "≥50"])
by_age = df.groupby(band, observed=True)[target].agg(["size", "mean"])
print(by_age.round(4).to_string())
rate_bars(by_age["mean"], by_age["size"], default_rate,
          "Taux de défaut par tranche d'âge", "eda_default_by_age.png")

- **Sexe** : F 7,0 % (202 448), M 10,1 % (105 059), **XNA : 4 lignes**, aucun défaut. XNA est exclu explicitement des calculs d'équité et compté dans le rapport. `CODE_GENDER` n'entre jamais dans le modèle.
- **Âge** : <30 ans **11,4 %**, 30-50 ans 8,7 %, ≥50 ans **5,7 %** : les moins de 30 ans font **2 fois** plus défaut que les 50 ans et plus. Comme `AGE_YEARS` est une feature, un écart brut d'acceptation selon l'âge est attendu : il est mesuré sur la baseline (notebook 02, D5).

## Conclusions

1. Cible déséquilibrée (8,07 %) : pas d'accuracy, pondération des classes.
2. 18 % de `DAYS_EMPLOYED` sentinelles (retraités) : NaN + indicateur.
3. Les `EXT_SOURCE_*` sont les meilleurs signaux mais souvent absents ; leur absence est informative.
4. Thin file (D10) : 11,9 % des dossiers, 9,95 % de défaut, 90 % de bons payeurs que le refus systématique perdrait.
5. Écarts de risque par sexe (3 pts) et âge (5,7 pts entre extrêmes) à surveiller dans l'audit d'équité.